In [ ]:
# ============================================================
# STEERING TRAINING — separable
# reasoning-guided activation steering
# ============================================================

import re
import random
from pathlib import Path

import torch
import torch.nn as nn
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


# ============================================================
# PATHS
# ============================================================
# ROOT_DIR is a root directory of the project. Put the path instead of "...".
ROOT_DIR = Path(r"...")

MODEL_DIR = ROOT_DIR / "models" / "MODEL" # choose the model from the models directory
DATA_PATH = ROOT_DIR / "data" / "datasets" / "separable" / "separable_train.xlsx" #choose short or full dataset

OUT_PATH = ROOT_DIR / "separable_train_full_RESULT.xlsx" #you can change the name and path of the output file or keep the default one


# ============================================================
# CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 0

MAX_STEPS = 10000 #choose the number of steps for the training: 10000 for full, 1000 for short
LR = 2e-4
GRAD_CLIP = 1.0
STEERING_ALPHA = 1.0

LAMBDA_WITH_REASONING = 0.7

MAX_REASONING_TOKENS = 2000
MAX_TOTAL_TOKENS = 4096

TORCH_DTYPE = torch.float16

GEN_MAX_NEW_TOKENS = 512
LOG_GEN_EVERY = 1


# ============================================================
# SEED
# ============================================================

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# NORMALIZATION + BLEU
# ============================================================

def normalize_answer(s: str) -> str:
    if not s:
        return ""

    s = re.sub(r"\\boxed\{(.+?)\}", r"\1", s)

    s = re.sub(r"^y\s*\(x\)\s*=\s*", "", s)
    s = re.sub(r"^y\s*=\s*", "", s)

    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace(" ", "")
    return s


def tokenize_math(expr: str):
    return re.findall(r"[A-Za-z]+|\d+|\^|\+|\-|\*|\/|\(|\)|\{|\}|C", expr)


def compute_bleu(true: str, pred: str) -> float:
    if not true or not pred:
        return 0.0
    return sentence_bleu(
        [tokenize_math(true)],
        tokenize_math(pred),
        weights=(0.5, 0.5),
        smoothing_function=SmoothingFunction().method1
    )


# ============================================================
# UTILS
# ============================================================

def ensure_boxed(ans: str) -> str:
    if r"\boxed{" in ans:
        return ans
    return r"\boxed{" + ans + "}"


def extract_boxed(text: str) -> str:
    matches = [m.start() for m in re.finditer(r"\\boxed\{", text)]
    if not matches:
        return ""
    start = matches[-1] + len(r"\boxed{")
    depth, i = 1, start
    while i < len(text) and depth:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1
    return text[start:i-1].strip()


def truncate_by_tokens(text: str, max_tokens: int, tokenizer) -> str:
    if not text:
        return ""
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    return tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)


# ============================================================
# PROMPT (POLYNOMIAL)
# ============================================================

BASE_SYS = """
You are a symbolic mathematics model.

Task: compute y(x) from the given derivative y'(x).

Output requirements:
- Output ONLY the final explicit separable equation y(x).
- Do NOT output integrals.
- Do NOT output the symbol \\int.
- Use LaTeX.
- Return exactly one boxed expression of the form \\boxed{y=...+C}.
- Include +C.
- No reasoning.
- No explanations.
"""


def make_prompt(eq: str, reasoning: str | None, include_reasoning: bool) -> str:
    p = BASE_SYS
    p += f"PROBLEM:\n{eq}\n\n"

    if include_reasoning and reasoning:
        p += f"REFERENCE SOLUTION:\n{reasoning}\n\n"

    p += "ANSWER:\n"
    return p


# ============================================================
# MODEL + TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=TORCH_DTYPE,
    device_map={"": 0} if DEVICE.type == "cuda" else None,
)

model.eval()
for p in model.parameters():
    p.requires_grad = False


# ============================================================
# STEERING MODULE
# ============================================================

class Steering(nn.Module):
    def __init__(self, model, alpha):
        super().__init__()
        self.layers = model.model.layers
        h = model.config.hidden_size
        self.vectors = nn.Parameter(torch.zeros(len(self.layers), h, device=DEVICE))
        self.alpha = alpha
        self.handles = []

    def install(self):
        self.remove()

        def make_hook(i):
            def hook(_, __, out):
                if isinstance(out, tuple):
                    out = out[0]
                return out + (self.alpha * self.vectors[i]).to(out.dtype)
            return hook

        for i, layer in enumerate(self.layers):
            self.handles.append(layer.mlp.down_proj.register_forward_hook(make_hook(i)))

    def remove(self):
        for h in self.handles:
            try:
                h.remove()
            except:
                pass
        self.handles = []


steering = Steering(model, STEERING_ALPHA)
steering.install()

optimizer = torch.optim.AdamW([steering.vectors], lr=LR)


# ============================================================
# DATA
# ============================================================

df = pd.read_excel(DATA_PATH)
df = df.dropna(subset=["equation", "true_answer"]).reset_index(drop=True)

print(f"Loaded polynomial dataset: {len(df)} samples")


# ============================================================
# LOSS
# ============================================================

def masked_ce_loss(prompt: str, target: str) -> torch.Tensor:
    target = target + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full = tokenizer(prompt + target,
                     add_special_tokens=False,
                     return_tensors="pt",
                     truncation=True,
                     max_length=MAX_TOTAL_TOKENS)

    input_ids = full["input_ids"].to(DEVICE)
    labels = input_ids.clone()
    labels[:, :min(len(prompt_ids), input_ids.shape[1] - 1)] = -100

    return model(input_ids=input_ids, labels=labels, use_cache=False).loss


def ce_nll(prompt: str, target: str) -> float:
    with torch.no_grad():
        return float(masked_ce_loss(prompt, target).item())


# ============================================================
# PAIRWISE COMPARISON
# ============================================================

@torch.no_grad()
def pairwise_compare(step, eq, true_boxed):

    prompt = make_prompt(eq, None, False)
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    old_alpha = steering.alpha

    # ---- NLL ----
    steering.alpha = 0.0
    nll_off = ce_nll(prompt, true_boxed)

    steering.alpha = STEERING_ALPHA
    nll_on = ce_nll(prompt, true_boxed)

    # ---- GENERATION ----
    def generate(alpha):
        steering.alpha = alpha
        seq = model.generate(
            **enc,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=False,
        )
        txt = tokenizer.decode(seq[0], skip_special_tokens=True)
        boxed = extract_boxed(txt)
        return txt, boxed

    raw_off, boxed_off = generate(0.0)
    raw_on,  boxed_on  = generate(STEERING_ALPHA)

    steering.alpha = old_alpha

    # ---- BLEU ----
    bleu_off = compute_bleu(normalize_answer(true_boxed), normalize_answer(boxed_off))
    bleu_on  = compute_bleu(normalize_answer(true_boxed), normalize_answer(boxed_on))

    # ---- PRINT ----
    print("\n" + "=" * 120)
    print(f"[PAIRWISE @ STEP {step}]")

    print("\nEQUATION:")
    print(eq)

    print("\nTRUE ANSWER:")
    print(true_boxed)

    #print("\n================ RAW MODEL OUTPUT ================")
    #print("\n--- OFF (alpha=0) ---")
    #print(raw_off)

    #print("\n--- ON (alpha=1) ---")
    #print(raw_on)

    print("\n================ EXTRACTED BOXED ANSWER ================")
    print("\n--- OFF (alpha=0) ---")
    print(boxed_off if boxed_off else "<EMPTY>")

    print("\n--- ON (alpha=1) ---")
    print(boxed_on if boxed_on else "<EMPTY>")

    print("\n================ METRICS ================")
    print(f"BLEU OFF: {bleu_off:.4f}")
    print(f"BLEU ON : {bleu_on:.4f}")
    print(f"NLL OFF : {nll_off:.4f}")
    print(f"NLL ON  : {nll_on:.4f}")
    print(f"ΔBLEU   : {bleu_on - bleu_off:+.4f}")
    print(f"ΔNLL    : {nll_off - nll_on:+.4f}")

    print("=" * 120 + "\n")


# ============================================================
# TRAIN LOOP
# ============================================================

for step in tqdm(range(1, MAX_STEPS + 1)):
    row = df.sample(1).iloc[0]

    eq  = str(row["equation"])
    ans = ensure_boxed(str(row["true_answer"]))

    reas_raw = row.get("solution_latex", "")
    reas = "" if pd.isna(reas_raw) else truncate_by_tokens(str(reas_raw), MAX_REASONING_TOKENS, tokenizer)

    loss_with = masked_ce_loss(make_prompt(eq, reas, True), ans)
    loss_wo   = masked_ce_loss(make_prompt(eq, None, False), ans)

    loss = LAMBDA_WITH_REASONING * loss_with + (1 - LAMBDA_WITH_REASONING) * loss_wo

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_([steering.vectors], GRAD_CLIP)
    optimizer.step()

    print(f"[STEP {step:04d}] loss={loss.item():.4f} | ||steer||={steering.vectors.norm().item():.4f}")

    if step % LOG_GEN_EVERY == 0:
        pairwise_compare(step, eq, ans)


# ============================================================
# SAVE
# ============================================================

torch.save(
    {
        "vectors": steering.vectors.detach().float().cpu(),
        "alpha": STEERING_ALPHA,
        "steps": MAX_STEPS,
    },
    OUT_PATH,
)

print("Separable steering training finished.")
